# TripoSG — heavy 3D prop generator (free Colab T4)

Generates ONE watertight 3D prop (a mossy rock for the Aetheria golem-fight clearing) from a single image, using **TripoSG** (VAST-AI-Research, MIT code + MIT weights, ~6 GB).

**Why TripoSG and not TRELLIS here:** TripoSG needs 8 GB VRAM — a free Colab T4 (16 GB) runs it comfortably. TRELLIS v1 needs **16 GB+** (verified A100/A6000), so a T4 sits exactly on its floor and risks OOM; TRELLIS stays for a bigger GPU (Kaggle 2×T4).

**Honest limits:** TripoSG is image-only (no text-to-3D) and geometry-only (no texture, no UVs — the rock gets its material in Blender/Godot). Output is a watertight manifold mesh, Y-up — rotate to Z-up before Godot import.

Steps: 1) check GPU, 2) clone + install, 3) inference → `mossy-rock.glb`, 4) decimate to ~10k faces for the game, 5) verify outputs.

In [1]:
# 1) GPU check — expect a T4 with ~16 GB
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

name, memory.total [MiB]
Tesla T4, 15360 MiB
cuda: True | device: Tesla T4


In [7]:
# 2) Clone TripoSG + install (weights ~6 GB download on first use)
# NOTE: upstream requirements pins numpy==1.22.3, which has no wheel for Colab Python 3.13 and fails to build, so install requirements without that pin and keep Colab numpy 2.x. --no-build-isolation is also needed: pip's isolated build env fails to build 'diso'.
%cd /content
!test -d TripoSG || git clone https://github.com/VAST-AI-Research/TripoSG.git
%cd /content/TripoSG
!grep -v '^numpy' requirements.txt > /tmp/req-no-numpy.txt
!pip install --no-build-isolation -r /tmp/req-no-numpy.txt -q
!pip install --no-build-isolation fast-simplification -q  # needed by cell 4 decimation (trimesh simplify_quadric_decimation)
print('install done')

/content
/content/TripoSG
install done


In [12]:
# 3) Inference: input-rock.png -> mossy-rock.glb
# NOTE: upstream TripoSG has no `TripoSG` class or `triposg.utils.remove_background` - the official API is TripoSGPipeline + BriaRMBG, as in scripts/inference_triposg.py, so run that script. PYTHONPATH is needed because running the script from scripts/ does not put the repo root on sys.path. The script is also patched so its custom scheduler/transformer/vae modules are imported before from_pretrained, as current diffusers requires.
%cd /content/TripoSG
!grep -q 'import triposg.schedulers.scheduling_rectified_flow' scripts/inference_triposg.py || sed -i "/from triposg.pipelines.pipeline_triposg import TripoSGPipeline/a import triposg.schedulers.scheduling_rectified_flow" scripts/inference_triposg.py
!grep -q 'import triposg.models.transformers.triposg_transformer' scripts/inference_triposg.py || sed -i "/import triposg.schedulers.scheduling_rectified_flow/a import triposg.models.transformers.triposg_transformer" scripts/inference_triposg.py
!grep -q 'import triposg.models.autoencoders.autoencoder_kl_triposg' scripts/inference_triposg.py || sed -i "/import triposg.models.transformers.triposg_transformer/a import triposg.models.autoencoders.autoencoder_kl_triposg" scripts/inference_triposg.py
!PYTHONPATH=/content/TripoSG python scripts/inference_triposg.py --image-input /content/input-rock.png --output-path /content/mossy-rock.glb --num-inference-steps 50 --guidance-scale 7.0 --seed 42
print('exported /content/mossy-rock.glb')

/content/TripoSG
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files: 100% 11/11 [00:00<00:00, 439.86it/s]
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B                         



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            



Fetching 20 files: 100% 20/20 [00:00<00:00, 517.34it/s]


Download complete:           |  0.00B            


Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights from local directory


Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 439/439 [00:00<00:00, 917.83it/s]


Loading pipeline components...:  20% 1/5 [00:00<00:02,  1.40it/s]

Loading pipeline components...: 100% 5/5 [00:01<00:00,  3.88it/s]
There are modules in TripoSGDiTModel that should be kept in f

In [17]:
# 4) Decimate to ~10k faces for the game (guarded: keeps the original if it fails)
# NOTE: trimesh 5.x simplify_quadric_decimation takes a reduction fraction positionally, so pass the target as face_count=10000 (needs fast-simplification, installed in cell 2).
import os
try:
    import trimesh
    m = trimesh.load('/content/mossy-rock.glb', force='mesh')
    print('faces before:', len(m.faces))
    if len(m.faces) > 10000:
        m = m.simplify_quadric_decimation(face_count=10000)
        m.export('/content/mossy-rock-10k.glb')
        print('faces after:', len(m.faces))
    else:
        print('already <= 10k faces, no decimation needed')
except Exception as e:
    print('decimation skipped:', e)

faces before: 1141932
faces after: 10000


In [18]:
# 5) Verify outputs
import os
for f in ['/content/mossy-rock.glb', '/content/mossy-rock-10k.glb']:
    print(f, os.path.getsize(f) if os.path.exists(f) else 'MISSING', 'bytes')
!ls -la /content/*.glb

/content/mossy-rock.glb 20556648 bytes
/content/mossy-rock-10k.glb 180776 bytes
-rw-r--r-- 1 root root   180776 Nov 15 11:18 /content/mossy-rock-10k.glb
-rw-r--r-- 1 root root 20556648 Nov 15 11:03 /content/mossy-rock.glb
